# A* vs MCTS (Design-based)
This notebook compares A* and MCTS **on Design.py only**.

## Imports

In [1]:

try:
    import os
    import glob
    import time
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

    import graphical_sampling as gs
    from package_sampling.utils import inclusion_probabilities
    from graphical_sampling.design import Design
    from graphical_sampling.criteria.var_nht import VarNHT
    from graphical_sampling.search.astar import AStar
    from graphical_sampling.search.mcts import MCTS
    from graphical_sampling.validation.design_validity import (
        check_probability_sum,
        check_fip_consistency,
    )
    print("Imported Successfully")
except KeyError as e:
    print(e)

Imported Successfully


## Load population MU284_filtered.csv


In [3]:
import os
import pandas as pd
import numpy as np

try:
    file_path = os.path.join('populations', 'MU284_filtered.csv')
    df = pd.read_csv(file_path)

    cs82 = df['CS82'].values.astype(float)  
    ss82 = df['SS82'].values.astype(float)  

    N = len(df)
    n = 20  
    rng = np.random.default_rng(42)

    print(f"✅ Data loaded. Population Size (N): {N}, Sample Size (n): {n}")

except FileNotFoundError:
    print("❌ Error: File 'MU284_filtered.csv' not found in the 'populations' folder.")
except KeyError as e:
    print(f"❌ Error: Column {e} not found in the CSV.")

✅ Data loaded. Population Size (N): 281, Sample Size (n): 20


## Select dataset

In [ ]:
probs = n * (cs82 / cs82.sum())

probs = np.minimum(probs, 1.0)

probs = probs * (n / probs.sum())

#print(probs)
print("✅ Inclusion probabilities constructed.")
print(f"   Sum of probs: {probs.sum():.5f} (Target: {n})")

✅ Inclusion probabilities constructed.
   Sum of probs: 20.00000 (Target: 20)


## Build initial Design (Design.py)

In [ ]:
import operator
import numpy as np
import time
from graphical_sampling.design import Design  

population_size = 10
initial_designs_list = []
criteria = operator.attrgetter('nht_variance')

print("=== PHASE 0: GREEDY DESCENT INITIALIZATION ===")
t0_init = time.perf_counter()


print("1. Creating Base Standard Design...")
standard_design = Design(inclusion=probs, variable=ss82, permute=False)
std_var = criteria(standard_design)
initial_designs_list.append(standard_design)
print(f"   Standard Design Variance: {std_var:.2f}")


print("2. Running 9 Greedy Walkers to dig deep valleys...")
for i in range(population_size - 1):

    current_design = standard_design.copy()
    current_var = std_var
    

    patience = 50 
    for step in range(patience):
        candidate = current_design.copy()
        
        rand_switch = np.random.uniform(0.5, 1.0)
        candidate.iterate(random_pull=False, switch_coefficient=rand_switch)
        

        cand_var = criteria(candidate)
        
        if cand_var <= current_var:
            current_design = candidate
            current_var = cand_var

    initial_designs_list.append(current_design)

t_init = time.perf_counter() - t0_init

init_vals = [criteria(d) for d in initial_designs_list]

print("\n--- Individual Variances of the Initial Population ---")
for idx, val in enumerate(init_vals):
    if idx == 0:
        print(f"Design {idx + 1:<2} (Standard) : {val:.2f}")
    else:
        print(f"Design {idx + 1:<2} (Greedy)   : {val:.2f}")
print("----------------------------------------------------\n")

initial_var = min(init_vals)

print(f"🔵 Initial Population Stats:")
print(f"   Best Variance:  {initial_var:.2f}")
print(f"   Mean Variance:  {np.mean(init_vals):.2f}")
print(f"   Init Time:      {t_init:.2f} s")

initial_design = standard_design

=== PHASE 0: GREEDY DESCENT INITIALIZATION ===
1. Creating Base Standard Design...
   Standard Design Variance: 241612.42
2. Running 9 Greedy Walkers to dig deep valleys...

--- Individual Variances of the Initial Population ---
Design 1  (Standard) : 241612.42
Design 2  (Greedy)   : 231889.22
Design 3  (Greedy)   : 228740.64
Design 4  (Greedy)   : 229755.91
Design 5  (Greedy)   : 239407.78
Design 6  (Greedy)   : 240108.81
Design 7  (Greedy)   : 235780.42
Design 8  (Greedy)   : 227845.50
Design 9  (Greedy)   : 232656.02
Design 10 (Greedy)   : 236753.67
----------------------------------------------------

🔵 Initial Population Stats:
   Best Variance:  227845.50
   Mean Variance:  234455.04
   Init Time:      0.74 s


## Run A* (Design-based)

In [ ]:
try:

    astar = AStar(
        initial_designs=[initial_design], 
        switch_coefficient=0.5,
        random_pull=False
    )
    
    t0 = time.perf_counter()
    

    astar.run(
        max_iterations=500,
        num_new_nodes=10,
        max_open_set_size=1000,
        num_changes=(1, 1),  
        top_k=5             
    )
    
    elapsed_astar = time.perf_counter() - t0

    print("A* best VarNHT:", astar.best_criteria_value)
    print("A* time (s):", round(elapsed_astar, 3))
    

except KeyError as e:
    print(e)
except TypeError as e:
    print(f"Correction needed: {e}")

A* best VarNHT: 236101.953125
A* time (s): 0.173


## Run MCTS (Design-based)

In [ ]:
import time

# =====================================================
# 1) Run Multi-Start MCTS (High Exploration Mode)
# =====================================================
t0 = time.perf_counter()

if __name__ == "__main__":
    mcts = MCTS(
        initial_designs=initial_designs_list,
        criteria=criteria,
        switch_coefficient= 1.0,  
        base_changes=20,          
        exploration_constant= 0.1 
    )

    best_design_mcts, best_value_mcts = mcts.run(
        max_iterations=150,    
        max_children_per_node=8
    )

elapsed_mcts = time.perf_counter() - t0

print("MCTS best VarNHT:", best_value_mcts)
print("MCTS time (s):", round(elapsed_mcts, 3))
# =====================================================
# 2) Validation — exactly what the professor asked
# =====================================================
print("\n=== VALIDATION: Best MCTS Design ===")

if 'best_design_mcts' in locals():
    target_design = best_design_mcts
elif 'best_design' in locals():
    target_design = best_design
else:
    raise NameError("❌ متغیر دیزاین پیدا نشد! لطفا سلول اجرای MCTS را دوباره اجرا کنید.")

target_probs = probs 

# ---- (A) Sum of probabilities = 1
ok_prob, prob_sum = check_probability_sum(target_design)

print(f"Probability sum: {prob_sum:.10f}")
print(f"Probability sum OK: {ok_prob}")

# ---- (B) FIP reconstruction check
ok_fip, max_diff, _ = check_fip_consistency(
    target_design,
    target_probs,
)

print(f"FIP consistency OK: {ok_fip}")
print(f"Max |FIP - reconstructed|: {max_diff:.10e}")


=== MULTI-START MCTS (Population Size: 10) ===
Best Initial Criteria: 227845.5
=== RUN STARTED (ADAPTIVE POPULATION MODE) ===

>>> ITER 0 | Depth 1 | Val: 0.00
  ★ BEST: 227845.5 -> 225902.2 (Type: high)
  ★ BEST: 225902.2 -> 221885.0 (Type: high)
  ★ BEST: 221885.0 -> 219597.2 (Type: low)

  >>> STAGNATION (8 iters) -> TELEPORT #1 to Best (219597.17)
  ★ BEST: 219597.2 -> 219583.6 (Type: high)
  ★ BEST: 219583.6 -> 219528.7 (Type: high)
  ★ BEST: 219528.7 -> 217688.8 (Type: high)
  ★ BEST: 217688.8 -> 216669.2 (Type: high)
  ★ BEST: 216669.2 -> 216474.8 (Type: high)
  ★ BEST: 216474.8 -> 216400.7 (Type: low)
  ★ BEST: 216400.7 -> 215877.9 (Type: high)

  >>> STAGNATION (8 iters) -> TELEPORT #2 to Best (215877.86)
  ★ BEST: 215877.9 -> 214920.8 (Type: high)
  ★ BEST: 214920.8 -> 211109.5 (Type: high)

  >>> STAGNATION (8 iters) -> TELEPORT #3 to Best (211109.52)

>>> ITER 20 | Depth 0 | Val: 0.00
  ★ BEST: 211109.5 -> 207935.0 (Type: high)
  ★ BEST: 207935.0 -> 198968.8 (Type: high)

 

## Final comparison

In [50]:
print("========== FINAL COMPARISON ==========")
print("Initial VarNHT :", initial_var)
print("A* VarNHT      :", astar.best_criteria_value)
print("MCTS VarNHT    :", best_value_mcts)


========== FINAL COMPARISON ==========
Initial VarNHT : 227845.5
A* VarNHT      : 236101.953125
MCTS VarNHT    : 133265.65625
